# Module 08 — Capstone: A Production Multi-Agent Strategy Engine with Strands

**Scenario — MarketNest (fictional e-commerce co.):** leadership must decide whether to launch a
*Premium Subscription Tier*. This notebook builds a coordinated team of specialist agents that turns a
**Situation Summary** into a defensible **Executive Strategy Briefing** (options A/B/C, tradeoffs,
risks, confidence, recommended path).

## Team & patterns

| Sub-agent              | Role                                                                                        | Pattern it demonstrates                    |
| ------------------------| ---------------------------------------------------------------------------------------------| --------------------------------------------|
| Planner                | Decomposes the decision into an analysis plan                                               | Swarm member                               |
| Researcher             | Extracts known facts / unknowns from the brief                                              | Swarm member                               |
| Risk Analyzer          | Risks + mitigations per option                                                              | Swarm member · runs **parallel** to demand |
| Market Demand Analyzer | Demand, fit, timing per option                                                              | Swarm member · runs **parallel** to risk   |
| Synthesizer            | Assembles the draft briefing                                                                | Swarm member                               |
| Reviewer               | **Critic** — finds gaps (correctness, hallucination, groundedness) and hands back for fixes | Swarm member / Critic                      |
| Evaluator              | Confidence score (0-100) + what would raise it                                              | Sequential stage                           |
| Writer                 | Final professional briefing                                                                 | Sequential stage                           |

## Orchestration

```
             +--------------------------- analysis_team (SWARM) ---------------------------+
 SITUATION   |  planner -> researcher -> (risk_analyzer / demand_analyzer) -> synthesizer  |
 SUMMARY ->  |                       ^                                  |                  |  -> evaluator -> writer -> FINAL
             |                       +---- reviewer (critic loop) ------+                  |     (sequential graph edges)
             +----------------------------------------------------------------------------+
```

The six analysis agents collaborate autonomously in a **Swarm** (handoffs + reviewer critic loop).
That Swarm is one node in a deterministic **Graph** whose sequential edges enforce
*review → evaluate → write*. The **parallel** pattern (independent Risk & Demand analysts) and the
**sequential**/**critic** patterns are each shown in standalone showcase cells below.


In [ ]:
# Install dependencies (takes ~20 seconds on first run)
%pip install -r requirements.txt

---

## Part-1: Define Model in Bedrock

In [ ]:
# The ONLY third-party dependency in this workshop is the Strands Agents SDK:
#     pip install strands-agents
# Swarm and GraphBuilder ship inside it, so no other external library is needed.
# Every agent reasons purely over the text we hand it (no web/tool calls), which is
# also our strongest guardrail against hallucinated facts.
from strands import Agent
from strands.models import BedrockModel
from strands.multiagent import Swarm, GraphBuilder

In [ ]:
# --- Model + shared configuration ----------------------------------------
# Claude profile enabled in your account (e.g. a Haiku profile for cheaper demos).
MODEL = BedrockModel(
    model_id="us.anthropic.claude-opus-5",
    max_tokens=16384,        # was defaulting low — this is what truncated the handoff
)

---

## Part-2: Sub-agent definitions

Each cell defines one specialist as a **builder function** returning a fresh `Agent`
(system prompt + tools). Builders let every showcase and the final run start from clean agents.


#### Planner Agent

In [ ]:
# --- PLANNER -------------------------------------------------------------
PLANNER_PROMPT = f"""
Role: Planner Agent for the MarketNest strategy team.

Input: Situation Summary.

Output: Exactly three sections:
• TASK_DECOMPOSITION (5-8 bullets) - each bullet is a concrete sub-task a strategy analyst would do, and names the teammate who owns it (researcher, risk_analyzer, demand_analyzer, synthesizer).
• MISSING_INFO (2-3 bullets) - inputs missing from the Situation Summary that are needed to decide responsibly.
• DECISION_CRITERIA (3-5 bullets) - criteria the recommendation should optimise for (e.g., conversion lift, unit economics, operational feasibility, customer trust).

Rules:
- In the output, each bullet point having 1 senetence with max 15 words.
- Do NOT recommend an option yet.
- No invented metrics; if a value is absent, note it under MISSING_INFO.
- Use crisp bullets, not paragraphs.
- After emitting the three sections, hand off to the 'researcher'.
"""

def build_planner():
    return Agent(model=MODEL, name="planner", system_prompt=PLANNER_PROMPT)

---
#### Reasearcher Agent

In [ ]:
# --- RESEARCHER ----------------------------------------------------------
RESEARCHER_PROMPT = f"""
Role: Researcher Agent for the MarketNest strategy team.

Inputs: Situation Summary; Planner output.

Output: Four sections:
• FACTS (top 2 bullets) - only facts explicitly stated in the Situation Summary.
• ASSUMPTIONS (top 2 bullets) - list assumptions separately (anything inferred, not stated).
• UNKNOWNS / QUESTIONS (top 2 bullets) - what must be answered to proceed.
• RISK_TRIGGERS (top 2 bullets) - conditions that would make this decision unsafe (e.g., fulfillment SLA unmet, unit economics negative, required approvals missing).

Rules:
- In the output, each bullet point having 1 senetence with max 15 words.
- If it is not explicitly stated in the Situation Summary, it cannot be a FACT.
- Do NOT recommend an option.
- No invented numbers.
- After emitting the four sections, hand off to 'risk_analyzer' and 'demand_analyzer'.
"""

def build_researcher():
    return Agent(model=MODEL, name="researcher", system_prompt=RESEARCHER_PROMPT)

---

#### Risk Analyzer Agent

In [ ]:
# --- RISK ANALYZER (runs in parallel with the demand analyzer) -----------
RISK_PROMPT = f"""Y
Role: Risk Analyzer Agent (specialist in the shared analyzer role).

Inputs: Situation Summary; Planner output; Researcher output.

Output: Risk Analysis with sections:
• Risk Summary (1-2 lines): the most decision-relevant risk theme.
• Risks by Option A/B (Max 2 options) - for each option, highlight (in 1 senetence with max 15 words) the material operational/fulfillment, financial/unit-economics, execution/timeline, and exit-criteria risks.
• Mitigations (2 bullets, mapped to the risks above).
• Risk Triggers (2 bullets) - conditions that would make the decision unsafe or trip the exit/kill criteria.
• Assumptions & Unknowns (2 bullets).

Rules:
- In the output, each bullet point having 1 senetence with max 15 words.
- Do not invent metrics or facts.
- If information is missing, say what you would measure/confirm (do not guess a number).
- Make trade-offs explicit ("We gain X; we risk Y").
- Do NOT recommend an option; provide the risk view only.
- When done, hand back to the 'synthesizer'.
"""

def build_risk_analyzer():
    return Agent(model=MODEL, name="risk_analyzer", system_prompt=RISK_PROMPT)

---
#### Market Demand Analyzer Agent

In [ ]:
# --- MARKET DEMAND ANALYZER (runs in parallel with the risk analyzer) ----
DEMAND_PROMPT = f"""Y
Role: Market Demand Analyzer Agent (specialist in the shared analyzer role).

Inputs: Situation Summary; Planner output; Researcher output.

Output: Demand Analysis with sections:
• Demand Summary (1-2 lines): the most decision-relevant demand signal.
• Demand by Option A/B (Max 2 options) - for each option, highlight (in 1 sentence with 15 max words) customer-segment fit, conversion potential against the stated success criteria, and market timing.
• Supporting Signals (2 bullets) - facts from the inputs that raise or lower demand confidence.
• Demand Risks (2 bullets) - where demand could fall short of the success criteria.
• Assumptions & Unknowns (2 bullets).

Rules:
- In the output, each bullet point having 1 senetence with max 15 words.
- Do not invent metrics or facts.
- If information is missing, say what you would measure/confirm (do not guess a number).
- Make trade-offs explicit ("We gain X; we risk Y").
- Do NOT recommend an option; provide the demand view only.
- When done, hand back to the 'synthesizer'.
"""

def build_demand_analyzer():
    return Agent(model=MODEL, name="demand_analyzer", system_prompt=DEMAND_PROMPT)

---
#### Information Synthesizer Agent

In [ ]:
# --- SYNTHESIZER ---------------------------------------------------------
SYNTHESIZER_PROMPT = f"""
Role: Synthesis Agent for the MarketNest strategy team.


Inputs: Market Demand Analysis (market/customer lens); Risk Analysis (risk lens); plus the Planner and Researcher outputs.

Output: A single Executive Strategy Briefing with sections: 
- Recommendation (1-2 lines)
- Options Considered A/B/C (3 Max)
- Trade offs (2 bullets)
- Risks & Mitigations (2 bullets)
- Assumptions & Unknowns (2 bullets)
- Escalate? (Yes/No + reason)
- Next Steps (owner + timeline - 2 bullets)"

Rules:
- In the output, each bullet point having 1 senetence with max 15 words.
- Do NOT concatenate the two analyses; synthesise them and resolve conflicts.
- Include at least two explicit conflict-resolution sentences using the pattern above.
- Do not invent facts or metrics; use "Unknown" where data is missing.
- If the reviewer returns gaps, revise only the flagged parts and hand back to the 'reviewer'.
- Resolve every demand-vs-risk conflict inline (≥2×): "Demand says ___. Risk says ___. We choose ___ because ___."
- When the draft is complete, hand off to the 'reviewer'.
"""

def build_synthesizer():
    return Agent(model=MODEL, name="synthesizer", system_prompt=SYNTHESIZER_PROMPT)

---

#### Reviewer (to critique) Agent

In [ ]:
# --- REVIEWER (the critic that gates quality) ----------------------------
REVIEWER_PROMPT = f"""
Role: Critic (red-team reviewer) for the MarketNest strategy team.

Input: Draft Executive Strategy Briefing (from the synthesizer).

Output (while gaps remain):
• CRITIQUES (top 2 bullets) - specific correctness/logic gaps.
• MISSING_INFO (top 2 bullets) - inputs needed but absent from the Situation Summary.
• HALLUCINATION_RISKS (top 2 bullets) - claims/metrics not traceable to the Situation Summary or an analyst's output.
• GROUNDEDNESS (top 2 bullets) - conclusions that outrun their evidence.
• ESCALATE (Yes/No + reason).
• EVIDENCE_TO_RESOLVE (1-2 bullets).

Rules:
- In the output, each bullet point having 1 senetence with max 15 words.
- Be tough and specific; assume the briefing is wrong until proven otherwise.
- Identify at least five specific issues while any gap remains.
- If exit/kill criteria, unit economics, or required approvals are not addressed, set Escalate=Yes.
- Do NOT rewrite the briefing. Hand off each issue to the teammate best able to fix it
  ('synthesizer' for structure/wording, 'risk_analyzer' or 'demand_analyzer' for domain gaps,
  'researcher' for missing facts).
- Only when the briefing is complete, grounded, and hallucination-free, reply beginning with
  "APPROVED:" plus a one-line rationale, then stop (no further handoff).
"""

def build_reviewer():
    return Agent(model=MODEL, name="reviewer", system_prompt=REVIEWER_PROMPT)

---

#### Evaluator (to produce a confidence score) Agent

In [ ]:
# --- EVALUATOR (confidence scoring, sequential stage) --------------------
EVALUATOR_PROMPT = f"""
Role: Confidence Scorer for the MarketNest strategy team.

Input: The approved Executive Strategy Briefing.

Output:
• BRIEFING (carried forward unchanged — do not edit its content).
• CONFIDENCE_SCORE (0-100) — how defensible and well-grounded the recommendation is given the available evidence.
• WHY (3 bullets) — the drivers behind the score, both strengths and weaknesses.
• WHAT_WOULD_INCREASE_CONFIDENCE (3 bullets) — ranked by impact.

Rules:
- In the output, each bullet point having 1 senetence with max 15 words.
- Lower the score whenever a key constraint (exit/kill criteria, unit economics, fulfillment SLA, required approvals) lacks supporting evidence.
- Each "increase confidence" item must be concrete and actionable (e.g., run a metro-zone pricing test, confirm next-day-delivery capacity at pilot volume, secure Finance sign-off on unit economics, size the pilot cohort for statistical power).
- Score the briefing; do not rewrite or re-argue it.
"""

def build_evaluator():
    return Agent(model=MODEL, name="evaluator", system_prompt=EVALUATOR_PROMPT)

---
#### Report Writer Agent

In [ ]:
# --- WRITER (final professional output, sequential stage) ----------------
WRITER_PROMPT = f"""
Role: Professional Writer for the MarketNest strategy team.

Input: The approved Executive Strategy Briefing and the Evaluator's Confidence Report.

Output: Assemble them into one final, paste-ready markdown artefact with these exact markers:
=== EXECUTIVE STRATEGY BRIEFING ===
BRIEFING (carried forward unchanged — do not edit its content)

=== CONFIDENCE ===
- CONFIDENCE_SCORE (0-100)
- WHY (3 bullets)
- WHAT_WOULD_INCREASE_CONFIDENCE (3 bullets)

Rules:
- In the output, each bullet point having 1-2 senetence with each senetence max 15 words.
- Preserve the briefing's 8-section structure and its content; polish wording only, do not re-argue or add facts.
- No extra sections beyond the two marked blocks.
- Clean, executive-ready markdown that pastes into a doc without fixup (consistent headers, tight bullets, no stray characters).
"""

def build_writer():
    return Agent(model=MODEL, name="writer", system_prompt=WRITER_PROMPT)

---

## Part-3: Define one reusable **`visualize_agents(...)`** helper, then follow each pattern with a diagram of how its agents call one another.

In [ ]:
# --- Reusable topology visualizer ----------------------------------------
# Draws an agent-call / handoff diagram from a list of edges and displays it inline.
#   edges : list of (src, dst) or (src, dst, kind); kind in {"forward","critic","sequential"}
#   groups: optional {node: group_label} to color nodes by role
#   box   : optional list of nodes to enclose in a dashed boundary (e.g. a Swarm)
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

_NODE_PALETTE = ["#4C78A8", "#54A24B", "#F58518", "#72B7B2", "#B279A2"]
_EDGE_STYLE = {
    "forward":    dict(edge_color="#9aa0a6", width=1.8, arrowsize=16, style="solid",  connectionstyle="arc3,rad=0.0"),
    "sequential": dict(edge_color="#1f77b4", width=2.6, arrowsize=20, style="solid",  connectionstyle="arc3,rad=0.0"),
    "critic":     dict(edge_color="#E45756", width=1.6, arrowsize=15, style="dashed", connectionstyle="arc3,rad=-0.32"),
}
_EDGE_LABEL = {"forward": "Handoff (forward)", "sequential": "Sequential edge", "critic": "Critic fix-loop"}

def _layered_pos(nodes, forward_edges):
    """Left-to-right layout: x = longest-path depth (parallel nodes share a column)."""
    H = nx.DiGraph(); H.add_nodes_from(nodes); H.add_edges_from(forward_edges)
    layer = {n: 0 for n in nodes}
    if nx.is_directed_acyclic_graph(H):
        for n in nx.topological_sort(H):
            for _, d in H.out_edges(n):
                layer[d] = max(layer[d], layer[n] + 1)
    columns = {}
    for n in nodes:
        columns.setdefault(layer[n], []).append(n)
    pos = {}
    for x, members in columns.items():
        members = sorted(members)
        k = len(members)
        for i, n in enumerate(members):
            pos[n] = (x * 1.7, (k - 1) / 2 - i)
    return pos

def visualize_agents(title, edges, groups=None, box=None, box_label="Swarm", pos=None, filename=None):
    """Render + display an agent topology; also saves a PNG. Returns the filename."""
    norm = [(e[0], e[1], e[2] if len(e) > 2 else "forward") for e in edges]
    nodes = sorted({n for e in norm for n in e[:2]})
    groups = groups or {n: "agent" for n in nodes}
    labels = sorted(set(groups.values()))
    color_of = {g: _NODE_PALETTE[i % len(_NODE_PALETTE)] for i, g in enumerate(labels)}
    pos = pos or _layered_pos(nodes, [(s, d) for s, d, k in norm if k != "critic"])

    G = nx.DiGraph(); G.add_nodes_from(nodes); G.add_edges_from([(s, d) for s, d, _ in norm])
    fig, ax = plt.subplots(figsize=(max(7, 1.9 * (max(x for x, _ in pos.values()) + 1) + 2), 4.8))

    if box:
        xs = [pos[n][0] for n in box]; ys = [pos[n][1] for n in box]; pad = 0.6
        ax.add_patch(mpatches.FancyBboxPatch(
            (min(xs) - pad, min(ys) - pad), (max(xs) - min(xs)) + 2 * pad, (max(ys) - min(ys)) + 2 * pad,
            boxstyle="round,pad=0.02", linestyle="--", linewidth=1.6,
            edgecolor="#6b7280", facecolor="#f3f4f6", zorder=0))
        ax.text(min(xs) - pad + 0.05, max(ys) + pad - 0.05, box_label,
                fontsize=9, color="#6b7280", weight="bold", va="bottom")

    handles = []
    for g in labels:
        gnodes = [n for n in nodes if groups.get(n) == g]
        nx.draw_networkx_nodes(G, pos, nodelist=gnodes, node_color=color_of[g], node_size=2000, ax=ax)
        handles.append(mpatches.Patch(color=color_of[g], label=g))
    # Wrap long names on "_" so labels fit inside the node.
    nx.draw_networkx_labels(G, pos, labels={n: n.replace("_", "\n") for n in nodes},
                            font_size=7, font_color="white", font_weight="bold", ax=ax)

    for kind in ("forward", "sequential", "critic"):
        el = [(s, d) for s, d, k in norm if k == kind]
        if el:
            nx.draw_networkx_edges(G, pos, edgelist=el, node_size=2000, ax=ax, **_EDGE_STYLE[kind])
            handles.append(mpatches.Patch(color=_EDGE_STYLE[kind]["edge_color"], label=_EDGE_LABEL[kind]))

    ax.legend(handles=handles, loc="lower center", ncol=len(handles),
              fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.12))
    ax.set_title(title, fontsize=13, weight="bold")
    ax.axis("off"); plt.tight_layout()

    plt.show()

---

## Part-4: Pattern showcase: Sequential vs Parallel vs Swarm vs Critic

Define a Test Situation Summary

In [ ]:
TEST_SITUATION_SUMMARY = "Largest stadium in the world"

---

Each cell below builds a minimal example of one orchestration pattern. Building agents/graphs is free (no model calls); the actual `.invoke` lines are **commented out** so you opt in per demo and avoid extra Bedrock cost. They use `TEST_SITUATION_SUMMARY`, defined in the final cell.

#### Sequential Pattern

In [ ]:
# SEQUENTIAL --------------------------------------------------------------
# Edges force strict order: 'researcher' runs only after 'planner' finishes and
# receives the planner's output as context.
seq_builder = GraphBuilder()
seq_builder.add_node(build_planner(), "planner")
seq_builder.add_node(build_researcher(), "researcher")
seq_builder.add_edge("planner", "researcher")          # planner -> researcher
seq_demo = seq_builder.build()

seq_result = seq_demo(TEST_SITUATION_SUMMARY)              
# print(seq_result)

In [ ]:
# Visualize the sequential pattern.
visualize_agents("Sequential Pattern",
                 [("planner", "researcher", "sequential")])

---

#### Parallel Pattern

In [ ]:
# PARALLEL ----------------------------------------------------------------
# 'risk_analyzer' and 'demand_analyzer' both depend on 'researcher' but NOT on each
# other, so Strands schedules them CONCURRENTLY after researcher completes (req. #4).
par_builder = GraphBuilder()
par_builder.add_node(build_researcher(), "researcher")
par_builder.add_node(build_risk_analyzer(), "risk_analyzer")
par_builder.add_node(build_demand_analyzer(), "demand_analyzer")
par_builder.add_edge("researcher", "risk_analyzer")     # fan-out
par_builder.add_edge("researcher", "demand_analyzer")   # fan-out -> runs parallel to risk
par_demo = par_builder.build()

par_result = par_demo(TEST_SITUATION_SUMMARY)              

In [ ]:
visualize_agents("Parallel Pattern",
                 [("researcher", "risk_analyzer", "forward"),
                  ("researcher", "demand_analyzer", "forward")],
                 groups={"researcher": "upstream",
                         "risk_analyzer": "parallel branch",
                         "demand_analyzer": "parallel branch"})

---

#### Swarn Pattern

In [ ]:
# SWARM -------------------------------------------------------------------
# Agents collaborate AUTONOMOUSLY via handoffs (Strands injects the handoff tools).
# Here the synthesizer drafts and the reviewer critiques; control bounces between them
# until the reviewer replies "APPROVED:".
swarm_demo = Swarm(
    [build_synthesizer(), build_reviewer()],
    max_handoffs=2,
    max_iterations=2,
    execution_timeout=600.0,
    node_timeout=180.0,
)

swarm_result = swarm_demo(TEST_SITUATION_SUMMARY)          

In [ ]:
# Visualize the swarm pattern (autonomous handoffs inside the dashed boundary).
visualize_agents("Swarm Pattern",
                 [("synthesizer", "reviewer", "forward"),
                  ("reviewer", "synthesizer", "critic")],
                 box=["synthesizer", "reviewer"],
                 box_label="Swarm (autonomous handoffs)")

---

#### Critic Pattern

In [ ]:
# CRITIC ------------------------------------------------------------------
# A DETERMINISTIC generate -> critique -> revise loop you drive from code (contrast with
# the Swarm, where handoffs are autonomous). Pure Strands SDK, no extra libraries.
def run_critic_demo(situation, rounds=1):
    generator = build_synthesizer()
    critic = build_reviewer()
    draft = result_text(generator(situation))
    for _ in range(rounds):
        critique = result_text(critic(f"Critique this draft briefing:\n\n{draft}"))
        if critique.strip().upper().startswith("APPROVED"):
            break
        draft = result_text(generator(
            f"Revise your draft to fix these review issues:\n{critique}\n\nDraft:\n{draft}"))
    return draft

def result_text(agent_result):
    """Pull plain text out of an AgentResult (or a graph/swarm node's .result)."""
    msg = getattr(agent_result, "message", None)
    if isinstance(msg, dict):
        return "\n".join(c.get("text", "") for c in msg.get("content", []) if "text" in c).strip()
    return str(agent_result)

critic_output = run_critic_demo(TEST_SITUATION_SUMMARY)    

In [ ]:
# Visualize the critic pattern (generator drafts, critic returns fixes).
visualize_agents("Critic Pattern",
                 [("synthesizer", "reviewer", "forward"),
                  ("reviewer", "synthesizer", "critic")],
                 groups={"synthesizer": "generator", "reviewer": "critic"})

---

## Part-5: Assemble the production system

The six analysis agents form the **Swarm** (`analysis_team`). That swarm is nested as a single node
in a **Graph** whose sequential edges run `analysis_team → evaluator → writer`.

In [ ]:
# --- Analysis Swarm: the 6 collaborating agents (req. #2 & #3) ------------
def build_analysis_swarm():
    researcher   = build_researcher()
    risk         = build_risk_analyzer()
    demand       = build_demand_analyzer()
    synthesizer  = build_synthesizer()
    reviewer     = build_reviewer()
    return Swarm(
        [researcher, risk, demand, synthesizer, reviewer],
        entry_point=researcher,                      # start with the planner
        max_handoffs=8,
        max_iterations=8,
        execution_timeout=900.0,
        node_timeout=300.0,
        repetitive_handoff_detection_window=2,    # guard against reviewer<->fixer ping-pong
        repetitive_handoff_min_unique_agents=2,
    )

# --- Full system: Swarm -> Evaluator -> Writer (sequential graph) ---------
def build_system():
    b = GraphBuilder()
    b.add_node(build_planner(), "planner")

    b.add_node(build_analysis_swarm(), "analysis_team")  # nested Swarm as one node
    b.add_edge("planner", "analysis_team")

    b.add_node(build_evaluator(), "evaluator")
    b.add_node(build_writer(), "writer")

    b.add_edge("analysis_team", "evaluator")   # evaluate only AFTER the reviewer approves (req. #5)
    b.add_edge("evaluator", "writer")          # write only AFTER scoring (req. #6)
    b.set_entry_point("planner")
    b.set_execution_timeout(1800)              # 30-min safety cap for the whole run
    return b.build()

In [ ]:
# --- INPUT + RUN ---------------------------------------------------------
SITUATION_SUMMARY = """Premium Subscription Tier Launch - MarketNest

We are considering launching a Premium Subscription Tier for MarketNest's top-performing product
line. The subscription would bundle free next-day delivery, a members-only rewards program, and
concierge customer support.

Customer segment & value delivered: Repeat buyers in metro zones (highest order frequency and
basket size). Value delivered is faster fulfillment, ongoing rewards, and premium support -
designed to deepen loyalty and lift purchase frequency.

Target dates / Target milestones: Decision needed within 2 weeks. Pilot launch targeted for Q3;
first membership cohort onboarded by end of Q3; mid-pilot review at week 6; go/no-go on full
rollout at month 3.

Success Criteria: Increase checkout conversion by 10% in metro zones within 3 months. Current
average order value (AOV) is approximately $86 (secondary watch metric).

Rollout with exit criteria: Phased rollout starting with a 5% metro-zone cohort, expanding to 25%
only after the mid-pilot review clears. Exit/kill criteria - halt and roll back if checkout
conversion lift is under 3% by week 6, if next-day delivery is met on fewer than 95% of member
orders, or if cost-to-serve per subscriber exceeds incremental margin.

Key questions: Can our fulfillment network sustain 95%+ next-day delivery in metro zones at pilot
volume? What subscription price point maximizes enrollment without eroding incremental margin?
Should we anchor on net-new subscribers or convert existing loyalty members first? Which metro
zones show the conversion headroom to hit the 10% lift?

Stakeholders: Chief Revenue Officer, Head of Growth Marketing, Supply Chain Lead, Customer
Experience Director.

Options: (A) Launch a paid Premium tier in metro zones anchored on the top 2 SKUs in Q3,
(B) Launch a lower-priced "Plus" tier (priority shipping only, no discounts) across a broader SKU
set to maximize enrollment, (C) Convert existing top loyalty-program members into Premium
subscribers before any net-new acquisition."""

# Build fresh, then run the full multi-agent system on the situation summary.
system = build_system()
final = system(SITUATION_SUMMARY)   # analysis_team (swarm) -> evaluator -> writer

print("Graph status :", final.status)
print("Executed nodes:", [n.node_id for n in final.execution_order])

print("\n===== EVALUATOR - CONFIDENCE & GAPS =====\n")
print(final.results["evaluator"].result)

print("\n===== FINAL EXECUTIVE STRATEGY BRIEFING =====\n")
print(final.results["writer"].result)

In [ ]:
# --- VISUALIZE the full system (reuses visualize_agents) -----------------
FULL_EDGES = [
    ("planner", "researcher", "forward"),
    ("researcher", "risk_analyzer", "forward"),
    ("researcher", "demand_analyzer", "forward"),
    ("risk_analyzer", "synthesizer", "forward"),
    ("demand_analyzer", "synthesizer", "forward"),
    ("synthesizer", "reviewer", "forward"),
    ("reviewer", "synthesizer", "critic"),      # critic loop: hand back to fix
    ("reviewer", "risk_analyzer", "critic"),
    ("reviewer", "demand_analyzer", "critic"),
    ("reviewer", "researcher", "critic"),
    ("reviewer", "evaluator", "sequential"),    # once APPROVED, the graph advances
    ("evaluator", "writer", "sequential"),
]
SWARM_MEMBERS = ["planner", "researcher", "risk_analyzer", "demand_analyzer", "synthesizer", "reviewer"]
FULL_GROUPS = {n: "swarm agent" for n in SWARM_MEMBERS}
FULL_GROUPS.update({"evaluator": "sequential stage", "writer": "sequential stage"})

visualize_agents("MarketNest Strategy Engine - Full Topology",
                 FULL_EDGES, groups=FULL_GROUPS,
                 box=SWARM_MEMBERS, box_label="Swarm (autonomous handoffs)")

---

## What You Built

A production-grade Decision Intelligence System that combines four multi-agent patterns:

| Pattern                 | Component                                       | Strands API                   |
| -------------------------| -------------------------------------------------| -------------------------------|
| Sequential (P1)         | Research phase: first, then analyze, then write | `GraphBuilder` edges          |
| Fork-Join/Parallel (P2) | Three option analyzers run simultaneously       | `GraphBuilder` parallel nodes |
| Critic-Refiner (P3)     | Quality gate on the final memo                  | `GraphBuilder` + cycle edge   |
| Swarm (P4)              | Subagent invokes each other on demand           | `@tool` wrapping `Agent`      |

---

## Key Takeaways

The complete system from input brief to approved leadership memo in ~50 seconds:
- 1 orchestrator (P5) coordinating 3 specialist tools
- 1 researcher calling 3 business intelligence tools (P1)
- 3 parallel option analyzers (P2)
- 1 writer + 1 critic in a quality loop (P3)

**Next:** Module 8 deploys this system to Amazon Bedrock AgentCore Runtime: managed compute, scalable endpoints, production observability.